In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

In [ ]:
DATA_DIR = Path("data_processed")
OUT_DIR = Path("artifacts/mlp")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

(X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape)

In [ ]:
feature_names = joblib.load(DATA_DIR / "feature_names.joblib")
scaler = joblib.load(DATA_DIR / "scaler.joblib")

print("Features:", len(feature_names))

In [ ]:
SEED = 42

# A bit more strict / regularized version:
# - early stopping on an internal validation split
# - stronger L2 regularization (alpha)
# - keep the same architecture for comparability
model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    learning_rate_init=1e-3,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    tol=1e-4,
    random_state=SEED,
)

model.fit(X_train, y_train)

def acc(X, y):
    return float(accuracy_score(y, model.predict(X)))

metrics = {
    "accuracy": {
        "train": acc(X_train, y_train),
        "val": acc(X_val, y_val),
        "test": acc(X_test, y_test),
    },
    "n_iter_": int(getattr(model, "n_iter_", -1)),
    "loss_": float(getattr(model, "loss_", np.nan)),
    "classes_": [int(c) for c in getattr(model, "classes_", [])],
    "seed": SEED,
    "mlp": {
        "hidden_layer_sizes": [100, 50],
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "max_iter": 2000,
        "early_stopping": True,
        "validation_fraction": 0.1,
        "n_iter_no_change": 20,
        "tol": 1e-4,
    },
    "data_dir": str(DATA_DIR),
}

metrics

In [ ]:
def predict_proba(X):
    return model.predict_proba(X)[:, 1]

In [ ]:
joblib.dump(model, OUT_DIR / "model.joblib")